# Linearized Fisher Updates for Continual Learning

This work studies continual adaptation of deep networks under constrained compute: small batches, limited accelerator memory, and a deliberately small trainable parameter subspace such as a Low Rank Adapter (LoRA) [1]. Elastic Weight Consolidation (EWC) [3] compresses earlier observations into a local quadratic approximation to their log likelihood. At time $t$, we represent that approximation by an anchor and a precision-like information summary,

$$ (\theta_t, \Lambda_t), \qquad \Lambda_t := n_t \widehat{\mathcal I}_t. $$

This pair resembles a Gaussian sufficient statistic, but it is only a local approximation in a general deep network. Its usefulness depends on keeping $\widehat{\mathcal I}_t$ coherent as the model moves through parameter space. A moving average incorporates new curvature observations but lags behind a changing parameter. The proposal here is to reduce that first-order lag with a **linearized Fisher update (LFU)**.

## Definitions and two coupled processes

Let $p_\theta(x)$ be a regular parametric model in a fixed parameter chart $\theta \in \Theta \subseteq \mathbb R^p$. Define

- the log likelihood $\ell(x;\theta) := \log p_\theta(x)$,
- the score $s(x;\theta) := \nabla_\theta \ell(x;\theta)$,
- the Fisher information matrix (FIM) $\mathcal I(\theta) := \mathbb E_\theta[s s^T]$, and
- a parameter displacement $u_t := \theta_{t+1}-\theta_t$.

For a negative log likelihood $L=-\ell$, the loss gradient is $g:=\nabla_\theta L=-s$. For a more general training loss, $g$ defines an empirical-Fisher or pseudo-score construction rather than the model Fisher; the distinction should be made explicit in each experiment.

Two related processes must not be conflated:

1. **Original learning process.** The optimizer, policy update, or adaptation rule uses incoming data to choose $u_t$ and moves $\theta_t$ to $\theta_{t+1}$. This notebook does not prescribe how $u_t$ is chosen; it assumes that the resulting steps are small enough for a local expansion to be useful.
2. **Auxiliary Fisher process.** Given the path produced by the original process, this process estimates the Fisher field along that path and updates $\widehat{\mathcal I}_t$ or $\Lambda_t$. It does not choose the destination. It consumes $u_t$ and asks how the information summary should change because of that move.

In short, the original process decides where the model moves; the auxiliary Fisher process tries to keep the compressed information summary coherent after the move. The auxiliary process is therefore well defined without inventing a Gaussian score family or assuming that it can be placed in canonical coordinates.

## Math sketch: the full Fisher derivative

For any sufficiently regular scalar or matrix-valued function $a(X,\theta)$,

$$ \partial_k \mathbb E_\theta[a(X,\theta)] = \mathbb E_\theta[\partial_k a(X,\theta) + a(X,\theta)s_k(X;\theta)]. $$

Applying this identity to $a=s_i s_j$ gives

$$ \partial_k \mathcal I_{ij}(\theta) = \mathbb E_\theta[(\partial_k s_i)s_j + s_i(\partial_k s_j) + s_i s_j s_k]. $$

Define the Amari-Chentsov tensor [2]

$$ C_{ijk}(\theta) := \mathbb E_\theta[s_i s_j s_k], $$

and define the **residual tensor**

$$ R_{ijk}(\theta) := \mathbb E_\theta[(\partial_k s_i)s_j + s_i(\partial_k s_j)]. $$

Using Einstein summation and $(T:u)_{ij}:=T_{ijk}u^k$, the directional derivative of the Fisher matrix is

$$ D\mathcal I_\theta[u] = (C_\theta+R_\theta):u. $$

The LFU is the first-order Taylor approximation

$$ \boxed{\mathcal I(\theta+u) = \mathcal I(\theta) + (C_\theta+R_\theta):u + O(\|u\|^2).} $$

For a canonical exponential family in its natural coordinates, $\partial_k s_i$ is deterministic and $\mathbb E_\theta[s_j]=0$, so $R_{ijk}=0$ and the Amari-Chentsov term is enough. A deep network in its ordinary weight coordinates is not generally in this case. Gaussianizing an estimator does not supply the unknown map from a model displacement $u$ to a canonical precision displacement, so it does not remove $R$.

The name residual tensor is operational. Strictly, the components $R_{ijk}$ are coordinate dependent under nonlinear reparameterization: they are a combination of connection-coefficient terms, whereas $C_{ijk}$ is an intrinsic tensor. Accordingly, an LFU is a coordinate-local Taylor update, not parallel transport. This is acceptable for the intended implementation, which remains in one fixed deep-network parameter chart or one fixed low-dimensional adapter chart.

## A matrix-free LFU estimator

Let

$$ h_u(X;\theta) := \nabla_\theta^2 \ell(X;\theta)u. $$

Contracting before taking expectations avoids materializing either rank-three array:

$$ D\mathcal I_\theta[u] = \mathbb E_\theta\left[h_u s^T + s h_u^T + (u^T s)ss^T\right]. $$

With the loss convention $L=-\ell$, $g=\nabla L$, and $H_u=\nabla^2 L\,u$, the same identity is

$$ \boxed{D\mathcal I_\theta[u] = \mathbb E_\theta\left[H_u g^T + gH_u^T - (u^T g)gg^T\right].} $$

Thus a sample LFU needs a per-sample gradient, one Hessian-vector product (HVP), and the scalar $u^Tg$. Reverse-mode autodiff computes $H_u$ by differentiating $g^Tu$; no dense Hessian is formed [4]. Moreover, with

$$ U=[g,H_u], \qquad B=\begin{bmatrix}-(u^Tg)&1\\1&0\end{bmatrix}, $$

the sample correction is $UBU^T$ and has rank at most two. An average of these corrections can be accumulated, sketched, truncated, or applied as a linear operator without storing $C$, $R$, or a dense Hessian. The correction is symmetric but need not be positive semidefinite; damping or projection may be needed after a finite first-order step.

## Computational regime

The proposal targets a specific regime:

- Only a manageable parameter subspace is adapted. Full-model LFUs are not assumed practical for large networks.
- Data arrive in small batches, possibly one observation at a time. Per-sample gradients must be retained or vectorized because an outer product of the batch-mean gradient is not the mean of per-sample outer products.
- HVPs are computed by autodiff. Relative to ordinary training, this adds roughly one backward-like pass and retains a higher-order graph, increasing both time and activation memory, but it does not store a $p\times p$ Hessian.
- Fisher summaries and LFU corrections use diagonal, block-diagonal, Kronecker-factored, low-rank, or sketched representations when a dense $p\times p$ matrix is too large.
- The model moves continuously enough that the omitted $O(\|u_t\|^2)$ term is controlled. Large steps should trigger re-estimation, subdivision into smaller LFUs, or rejection of the linear approximation.

Large sample limits may justify separate Gaussian or SDE models of the original learning process, but they are not needed for the LFU identity itself.

## EWC as local information compression

Suppose old observations are summarized at $\theta_t$ by $\Lambda_t=n_t\widehat{\mathcal I}_t$. For a candidate parameter $\vartheta$, their log likelihood is approximated by

$$ \log p(X_{\mathrm{old}};\vartheta) \approx \mathrm{const} - \frac12(\vartheta-\theta_t)^T\Lambda_t(\vartheta-\theta_t). $$

If the new-data likelihood is also approximated near its local MLE $\widehat\theta_{\mathrm{new}}$ with precision $\Lambda_{\mathrm{new}}$, then the EWC-regularized local MLE has the Gaussian product form

$$ \boxed{\widehat\theta_{t+1}=(\Lambda_t+\Lambda_{\mathrm{new}})^{-1}(\Lambda_t\theta_t+\Lambda_{\mathrm{new}}\widehat\theta_{\mathrm{new}}).} $$

The combined precision and natural parameter are

$$ \Lambda_{t+1}=\Lambda_t+\Lambda_{\mathrm{new}}, \qquad \eta_{t+1}=\Lambda_t\theta_t+\Lambda_{\mathrm{new}}\widehat\theta_{\mathrm{new}}. $$

These equations are exact for the two quadratic approximations. They reveal the most economical representation of accumulated quadratic evidence: Gaussian natural parameters $(\Lambda,\eta)$ add. Re-anchoring the same fixed quadratic does not require an LFU and does not change $\Lambda$; its center is recovered by solving $\Lambda\theta=\eta$. In a low-rank or singular trainable subspace, that solve requires damping, a pseudoinverse, or a structured solver.

This exposes an important target distinction. An LFU predicts the **model Fisher field** under the moving model distribution,

$$ \widehat{\mathcal I}_{t\to t+1}^{\mathrm{LFU}} := \widehat{\mathcal I}_t + \widehat{D\mathcal I_{\theta_t}[u_t]}. $$

It does not exactly update the observed information of a fixed old dataset. The latter is $-\nabla^2_\theta\log p(X_{\mathrm{old}};\theta)$, whose change at fixed $X_{\mathrm{old}}$ involves third derivatives of that old-data likelihood rather than $D\mathcal I_\theta[u]$. LFUs are therefore coherent for maintaining an estimate of current local model Fisher, which can shape future EWC factors as the agent traverses the manifold. Using an LFU to evolve the curvature of an already-compressed old-data factor is a further adaptive-EWC approximation and must be tested as such.

## Experimental considerations

LFUs trade moving-average lag for derivative-estimation error. The Amari-Chentsov contribution is a high-variance third score moment, while the residual contribution requires noisy HVPs. Although each contracted sample update is low rank, many updates can accumulate rank and may require truncation. The first-order approximation can also lose positive semidefiniteness or fail when $u_t$ is too large.

The first MNIST experiment should compare four Fisher estimators against frequent re-estimation at the current parameter:

1. an exponential moving average (EMA) baseline,
2. an Amari-Chentsov-only update $C:u_t$,
3. the full LFU $(C+R):u_t$, and
4. periodic fresh Fisher estimates as a higher-compute reference.

Useful ablations include HVP frequency, batch size down to one, adapter dimension, low-rank budget, damping, step size, and subdivision of large steps. Evaluation should measure Fisher approximation error, retained-task performance, adaptation to the changing task, wall-clock time, and peak memory. The central diagnostic is whether the residual tensor materially improves prediction of $\mathcal I(\theta_t+u_t)$ over both EMA and the Amari-Chentsov-only correction.

A second experiment should test whether the same conclusions survive structured or low-rank summaries at a scale closer to the intended edge-robotics setting. A final robotics demonstration can then test whether better-maintained EWC summaries improve continual adaptation of a vision-language model without replaying the original large dataset.

## A model of the original learning process

The LFU construction deliberately leaves $u_t$ unspecified. For numerical experiments, one may separately model the original process as tracking a slowly changing data-generating distribution. Let $P_t$ denote the environment at time $t$, let incoming observations satisfy $X_t\sim P_t$, and let an adaptation rule $A_t$ produce

$$ u_t=A_t(\theta_t,X_t,\text{optimizer state}), \qquad \theta_{t+1}=\theta_t+u_t. $$

This separates three objects: environmental change in $P_t$, the learning rule that determines $u_t$, and the auxiliary Fisher process driven by that realized $u_t$. Model correctness, independence, and slow change can be imposed as experiment-specific assumptions rather than being built into the definition of LFU.

For the theoretically convenient mixture interpretation of EWC, introduce an observed Bernoulli label $M_i$ with $\mathbb P(M_i=1)=\pi$, where $M_i=0$ marks old-task observations and $M_i=1$ marks new-task observations. Expanding the old-data likelihood around its MLE $\theta_t$ gives

$$ \frac1n\log p(X;\vartheta) \approx \frac1n\log p(X^{t+1};\vartheta)-\frac{1-\pi}{2}(\vartheta-\theta_t)^T\mathcal I(\theta_t)(\vartheta-\theta_t)+\mathrm{const}. $$

This recovers the EWC penalty as a local frequentist approximation. It does not make $(\theta_t,n_t\widehat{\mathcal I}_t)$ a globally sufficient statistic, and its accuracy must be checked as the anchor moves.

## Optional diffusion approximation

A diffusion model can be useful for studying the original process, but it is an additional asymptotic model rather than a consequence of LFU. The following display is the fixed-total Bernoulli-composition theory model. For a triangular array with many observations per small update, suppose the conditional increment satisfies

$$ \theta_{k+1,n}-\theta_{k,n}\approx \frac{\pi b(\theta_{k,n})}{n}+\sqrt{\frac{\pi}{n}}\,\mathcal I^{-1/2}(\theta_{k,n})\xi_k, \qquad \xi_k\sim_{iid}\mathcal N(0,I_p). $$

Under the usual regularity, tightness, and Lipschitz assumptions, scaling $k=\lfloor nt\rfloor$ suggests

$$ d\Theta_t=\pi b(\Theta_t)dt+\sqrt{\pi}\,\mathcal I^{-1/2}(\Theta_t)dW_t. $$

Here $b$ and $\pi$ describe how the original process moves. The auxiliary process estimates or predicts the Fisher term along the resulting path. The applied fixed-batch model developed below instead has innovation covariance proportional to $\pi^2/m$ and hence a noise coefficient linear in $\pi$. Experiments should not use either diffusion approximation as evidence that the residual tensor vanishes. The appendix **Diffusion and controlled small-noise limits** gives the triangular-array assumptions, conditional-moment arguments, and time scalings behind both models.

## Why MNIST is comparable to continual reinforcement learning

Both settings can be represented as adaptation to a slowly changing stream. In reinforcement learning, policy updates change the state-action distribution observed by the agent. In the proposed MNIST experiment, the frequency of digit 9 increases gradually after an initial fit on digits 0 through 8. In both cases, the original process follows an optimizer-dependent path $\theta_t$, while the auxiliary process tries to maintain curvature information along that path.

The comparison is intentionally limited: MNIST does not reproduce temporal credit assignment, policy-dependent sampling, or nonstationary transition dynamics. It isolates the narrower question of whether LFUs improve compressed Fisher tracking under controlled distribution shift.


## Single-observation batches

Single-observation updates reduce activation memory and fit the intended online setting, but they do not remove the need for per-sample derivatives. The useful scheduling principle is that each observation corrects the auxiliary Fisher process for the preceding parameter move, then helps the original learning process choose the next move.

### Directional LFU estimator

Suppose the preceding move was $u_{t-1}:=\theta_t-\theta_{t-1}$. After arriving at $\theta_t$, draw $X_t\sim p_{\theta_t}$ and evaluate

$$ s_t:=\nabla_\theta\ell(X_t;\theta_t), \qquad h_t:=\nabla_\theta^2\ell(X_t;\theta_t)u_{t-1}. $$

The lagged single-sample LFU contribution is

$$ \widehat\Delta_t^{\mathrm{lag}}=h_t s_t^T+s_t h_t^T+(u_{t-1}^Ts_t)s_t s_t^T. $$

For a negative log likelihood, let $g_t:=\nabla_\theta L(X_t;\theta_t)$ and $H_t:=\nabla_\theta^2L(X_t;\theta_t)u_{t-1}$. Then

$$ \widehat\Delta_t^{\mathrm{lag}}=H_tg_t^T+g_tH_t^T-(u_{t-1}^Tg_t)g_tg_t^T. $$

### Online LFU recursion

First use the lagged LFU to predict the current Fisher, then blend that prediction with direct curvature evidence at $\theta_t$. In the unified composition paradigm, the same $\pi_t$ that weights new likelihood information in the original learning process is the new-information weight, or forgetting rate, in the auxiliary process:

$$ \widetilde{\mathcal I}_t=\widehat{\mathcal I}_{t-1}+\widehat\Delta_t^{\mathrm{lag}}, $$

$$ \boxed{\widehat{\mathcal I}_t=(1-\pi_t)\widetilde{\mathcal I}_t+\pi_t Z_t}, \qquad Z_t:=s_ts_t^T. $$

The same observation $X_t$ may then help the original learning process choose $u_t$, after which $\theta_{t+1}=\theta_t+u_t$. Thus $X_t$ corrects the Fisher estimate for the past direction $u_{t-1}$ and creates the future direction $u_t$. A large $\pi_t$ both favors the new likelihood in the EWC objective and replaces more of the LFU-predicted Fisher with direct evidence; $1-\pi_t$ is the retained old-information weight. For the conditional calculations below, $\pi_t$ must be predictable with respect to the random quantities it weights. A recommendation calculated from the same observation it weights is an approximation whose selection bias must be assessed explicitly. The applied fixed-batch section later defines the shared scalar composition state induced by these same weights.

### Lagged directions and recursive error control

Let $\mathcal F_{t-1}$ contain the history after the move to $\theta_t$ but before drawing $X_t$. Then $\theta_t$ and $u_{t-1}$ are $\mathcal F_{t-1}$-measurable. Under conditionally on-model sampling,

$$ \mathbb E\left[\widehat\Delta_t^{\mathrm{lag}}\mid\mathcal F_{t-1}\right]=D\mathcal I_{\theta_t}[u_{t-1}]. $$

This predictability is what removes the same-sample coupling bias. In contrast, pairing $X_t$ with a direction $u_t(X_t)$ generally makes the sample LFU conditionally biased; merely computing the LFU after the optimizer step does not change their shared randomness.

The lagged estimator evaluates the derivative at the endpoint of the preceding move. If $D\mathcal I$ is locally Lipschitz with constant $L$, then

$$ \mathcal I(\theta_t)=\mathcal I(\theta_{t-1})+D\mathcal I_{\theta_t}[u_{t-1}]+r_t, \qquad \|r_t\|\leq\frac{L}{2}\|u_{t-1}\|^2. $$

Define the Fisher estimation error $e_t:=\widehat{\mathcal I}_t-\mathcal I(\theta_t)$, LFU noise $\varepsilon_t:=\widehat\Delta_t^{\mathrm{lag}}-D\mathcal I_{\theta_t}[u_{t-1}]$, and direct-observation noise $\zeta_t:=Z_t-\mathcal I(\theta_t)$. The $\pi_t$-weighted recursion gives

$$ e_t=(1-\pi_t)(e_{t-1}+\varepsilon_t-r_t)+\pi_t\zeta_t. $$

The convex recursion therefore prevents old errors and second-order remainders from accumulating as an uncontracted sum. For constant composition $\pi\in(0,1]$ and uniformly bounded remainder,

$$ \|\mathbb E[e_t]\|\leq(1-\pi)^t\|\mathbb E[e_0]\|+\frac{1-\pi}{\pi}\sup_j\|r_j\|. $$

For step size $\|u_t\|=O(\eta)$, the persistent truncation contribution is consequently $O((1-\pi)\eta^2/\pi)$ rather than an indefinitely growing $O(t\eta^2)$ drift. Recursive weighting controls this error but does not erase it. The composition also mediates a variance tradeoff: $\varepsilon_t$ enters the prediction with coefficient $1-\pi_t$, so an extremely small $\pi_t$ retains LFU noise for many iterations even while it smooths the direct observations $Z_t$. Because $\widehat\Delta_t^{\mathrm{lag}}$ and $Z_t$ use the same $X_t$, their correlation affects variance but not the displayed conditional means when $\pi_t$ is predictable.

These claims require the conditional distribution of $X_t$ to match the distribution defining $\mathcal I(\theta_t)$. Off-model data, uncorrected task drift, or temporal dependence beyond the conditioning state can introduce additional bias; reinforcement-learning experiments will need an appropriate conditional-Fisher or mixing interpretation.

### Autodiff implementation shape

For each sample, compute $g_t=\nabla L_t$ with a differentiable backward graph, form the scalar $g_t^Tu_{t-1}$, and differentiate that scalar once more to obtain $H_t$. The lagged LFU can then be stored as the two-column factorization

$$ U_t=[g_t,H_t], \qquad \widehat\Delta_t^{\mathrm{lag}}=U_t\begin{bmatrix}-(u_{t-1}^Tg_t)&1\\1&0\end{bmatrix}U_t^T. $$

The first gradient can be detached and reused by the optimizer when choosing $u_t$. This gives the lagged schedule a computational benefit: one observation and one gradient support both the previous LFU correction and the next learning update. In PyTorch, the extra HVP is primarily an additional reverse-mode pass; higher-order autodiff retains the forward graph and creates a graph for the first derivative, so peak memory can rise substantially even though activations are not simply duplicated and no dense Hessian is stored.

Because an LFU is a signed correction, a low-rank implementation should preserve signed factors rather than force every increment into a positive-semidefinite outer product. Positive semidefiniteness is a property to enforce on the resulting Fisher estimate, for example by damping, eigenvalue clipping in the maintained subspace, or a positive structured parameterization.


## Stochastic control: theoretical and applied models

Let $\pi_t\in[0,1]$ regulate adaptation to the population displacement $d\theta_t:=\theta_{t+1}^\star-\theta_t^\star$. In the unified paradigm it is both the new-observation weight in the original learning process and the direct-evidence weight in the auxiliary Fisher recursion. Two local experiments give different noise laws. The first is theoretically convenient and remains useful as an oracle. The second fixes the number of new observations and models the intended application.

### Euclidean parameter error and Fisher risk

Euclidean parameter MSE and local predictive divergence are different objectives. For a regular likelihood with nonsingular Fisher information,

$$ 2D_{\mathrm{KL}}(P_{\theta}\,\|\,P_{\theta+v})=v^T\mathcal I(\theta)v+o(\|v\|^2). $$

Thus $\|v\|^2$ measures coordinate displacement, while $\|v\|_{\mathcal I(\theta)}^2:=v^T\mathcal I(\theta)v$ measures local Fisher risk, equivalently second-order KL loss. Write $\mathsf M_t=I_{\dim\Theta}$ for Euclidean risk or $\mathsf M_t=\mathcal I(\theta_t^\star)$ for population Fisher risk, and set $\|v\|_{\mathsf M_t}^2=v^T\mathsf M_tv$. The derivations below use this metric at the population level. Its predictable finite-sample realization is a separate statistical construction in Appendix B. Optimization, LFU remainder control, and numerical displacement still occur in one fixed Euclidean parameter chart.

### Theoretical Bernoulli-composition model

Fix a predictable total stencil size $N$ and independently allocate each local likelihood contribution to the new point with probability $\pi_t$. Conditional on an oracle-recentered old summary, the new count is approximately $\pi_tN$, the combined Hessian is $N\mathcal I_t$, and the update covariance is $\pi_t\mathcal I_t^{-1}/N$. Define

$$ S_t^{(\mathsf M)}:=\|d\theta_t\|_{\mathsf M_t}^2, \qquad V_t^{(\mathsf M)}:=\operatorname{tr}(\mathsf M_t\mathcal I_t^{-1}). $$

The conditional one-step risk is

$$ R_{A,t}^{(\mathsf M)}(\pi)=(1-\pi)^2S_t^{(\mathsf M)}+\frac{\pi}{N}V_t^{(\mathsf M)}, $$

whose local surrogate minimizer is

$$ \boxed{\pi_{A,t}^{\mathrm{loc}}=\operatorname{clip}_{[0,1]}\left(1-\frac{V_t^{(\mathsf M)}}{2NS_t^{(\mathsf M)}}\right).} $$

At $S_t^{(\mathsf M)}=0$, define the minimizer by its limiting value $\pi_{A,t}^{\mathrm{loc}}=0$. Euclidean risk gives $V_t^{(\mathsf M)}=\operatorname{tr}\mathcal I_t^{-1}$; population Fisher risk gives $V_t^{(\mathsf M)}=\dim\Theta$. At the LAN scale $d\theta=b/\sqrt N$, the matching finite-information interpolation is

$$ d\Theta_t^{(\varepsilon_N)}=\pi_t b(\Theta_t^{(\varepsilon_N)})dt+\sqrt{\varepsilon_N\pi_t}\,\mathcal I(\Theta_t^{(\varepsilon_N)})^{-1/2}dW_t, \qquad \varepsilon_N=N^{-1/2}. $$

The $\sqrt{\pi_t}$ noise coefficient records that changing $\pi_t$ changes how many new random scores enter a fixed-total experiment. Appendix A supplies the triangular-array construction.

### Applied fixed-batch EWC model

In the applied experiment, a fixed batch of $m_t$ new observations arrives regardless of $\pi_t$. This fixed-batch process, rather than the preceding Bernoulli process, carries the composition state used below.

#### Composition state and timing

The principal symbols are:

| symbol | meaning at transition $t$ |
|---|---|
| $m_t$ | number of new observations that arrive |
| $\pi_t$ | chosen total weight on those new observations |
| $q_t$ | pre-transition concentration of historical influence weights |
| $N_{\mathrm{eff},t}$ | weight-based effective sample size $q_t^{-1}$ |
| $c$ | constant historical value of $\pi_s^{\mathrm{used}}$ in the stationary submodel |

Before transition $t$, let $w_{t,i}$ be the normalized conceptual influence weights represented by the compressed summary. They need not be materialized. Define

$$ \sum_iw_{t,i}=1, \qquad q_t:=\sum_iw_{t,i}^2, \qquad N_{\mathrm{eff},t}:=q_t^{-1}. $$

The timing is

$$ \underbrace{q_t}_{\text{known before the decision}}\;\xrightarrow[\text{fixed batch }m_t]{\text{choose }\pi_t}\;\underbrace{q_{t+1}}_{\text{next composition state}}. $$

Retaining each old weight as $(1-\pi_t)w_{t,i}$ and assigning each new observation weight $\pi_t/m_t$ gives the exact scalar recursion

$$ \boxed{q_{t+1}=(1-\pi_t)^2q_t+\frac{\pi_t^2}{m_t}.} $$

The identity is deterministic given the realized composition history; it requires no independence assumption and introduces no estimated matrix. Because the original and auxiliary processes consume the same $\pi_t$, they share this nominal weight bookkeeping when initialized from the same weights. Interpreting $q_t$ as parameter covariance still requires the calibrated-summary assumption below.

For the stationary-policy submodel, let $c\in(0,1]$ denote the constant historical action and let $m$ denote the constant batch size:

$$ \pi_s^{\mathrm{used}}\equiv c, \qquad m_s\equiv m. $$

The symbol $c$ is not another controller or hyperparameter. It distinguishes the past action that generated the memory state from the candidate action $\pi$ optimized at the next decision. The scalar recursion has equilibrium

$$ q_\infty(c)=\frac{c}{m(2-c)}, \qquad N_{\mathrm{eff},\infty}(c)=\frac{m(2-c)}{c}. $$

Under adaptive composition, no constant $c$ is assumed: the learner simply carries $q_t$ forward. The retained weight of information added at step $s$ is $\prod_{j=s+1}^{t-1}(1-\pi_j)$ before transition $t$, so there is no separate fixed half-life parameter.

#### Local tracking risk

The compressed old likelihood and mean new likelihood are combined as

$$ Q_t(\vartheta)=(1-\pi_t)Q_{\mathrm{old},t}(\vartheta)+\pi_tQ_{\mathrm{new},t}(\vartheta). $$

Under matched local quadratic curvature,

$$ \widehat\theta_{t+1}\approx(1-\pi_t)\widehat\theta_t+\pi_t\widehat\theta_{\mathrm{new},t}. $$

Let $e_t:=\widehat\theta_t-\theta_t^\star$ and $\epsilon_{t+1}:=\widehat\theta_{\mathrm{new},t}-\theta_{t+1}^\star$. Then

$$ \boxed{e_{t+1}=(1-\pi_t)(e_t-d\theta_t)+\pi_t\epsilon_{t+1}.} $$

Let $\mathcal E_t$ describe the replicated local experiment, including the population state and design but not the realized learner anchor. A calibrated-summary model assumes

$$ \mathbb E[e_t\mid\mathcal E_t]=\mu_t, \qquad \operatorname{Cov}(e_t\mid\mathcal E_t)=q_tK_t^{\mathrm{old}}, $$

$$ \mathbb E[\epsilon_{t+1}\mid\mathcal E_t]=0, \qquad \operatorname{Cov}(\epsilon_{t+1}\mid\mathcal E_t)=\frac{K_t^{\mathrm{new}}}{m_t}, $$

with centered old and new errors conditionally uncorrelated. The scalar $q_t$ is the pre-transition weight concentration defined above; interpreting it as a covariance multiplier is an assumption, not a consequence of the weight identity. Define

$$ S_t^{(\mathsf M)}:=\|d\theta_t-\mu_t\|_{\mathsf M_t}^2, \qquad D_t^{\mathrm{old}}:=\operatorname{tr}(\mathsf M_tK_t^{\mathrm{old}}), \qquad D_t^{\mathrm{new}}:=\operatorname{tr}(\mathsf M_tK_t^{\mathrm{new}}). $$

The one-step risk marginal over learner histories, conditional on $\mathcal E_t$, is

$$ R_{B,t}^{\mathrm{marg}}(\pi)=(1-\pi)^2\left(S_t^{(\mathsf M)}+q_tD_t^{\mathrm{old}}\right)+\pi^2\frac{D_t^{\mathrm{new}}}{m_t}. $$

Its exact local minimizer is

$$ \boxed{\pi_{B,t}^{\mathrm{marg}}=\frac{S_t^{(\mathsf M)}+q_tD_t^{\mathrm{old}}}{S_t^{(\mathsf M)}+q_tD_t^{\mathrm{old}}+D_t^{\mathrm{new}}/m_t}.} $$

This is not the same conditioning problem as acting from one realized learner. If $\mathcal F_t$ fixes the current anchor and hence $e_t$, then

$$ R_{B,t}^{\mathrm{cond}}(\pi)=(1-\pi)^2\|d\theta_t-e_t\|_{\mathsf M_t}^2+\pi^2\frac{D_t^{\mathrm{new}}}{m_t}, $$

$$ \boxed{\pi_{B,t}^{\mathrm{cond}}=\frac{\|d\theta_t-e_t\|_{\mathsf M_t}^2}{\|d\theta_t-e_t\|_{\mathsf M_t}^2+D_t^{\mathrm{new}}/m_t}.} $$

The conditional target omits $q_tD_t^{\mathrm{old}}$ because old-anchor uncertainty has become a realized displacement. One trajectory can measure this oracle displacement when $\theta_t^\star$ is available, but it cannot separately identify the replica mean $\mu_t$ and covariance calibration.

#### Covariance-only recommendation

In the centered, locally stationary subproblem, $\mu_t=0$ and $d\theta_t=0$. The true covariance-only composition is the population estimand

$$ \pi_t^{\mathrm{cov}}:=\underset{\pi\in[0,1]}{\arg\min}\;\mathbb E\!\left[\|\widehat\theta_{t+1}(\pi)-\theta^\star\|_{\mathsf M_t}^2\,\middle|\,\mathcal E_t\right], $$

and the preceding risk gives

$$ \boxed{\pi_t^{\mathrm{cov}}=\frac{q_tD_t^{\mathrm{old}}}{q_tD_t^{\mathrm{old}}+D_t^{\mathrm{new}}/m_t}.} $$

#### Equal-shape and stationary simplifications

When the local covariance shapes match in the chosen metric, $D_t^{\mathrm{old}}\approx D_t^{\mathrm{new}}$, the matrix terms cancel and give the tracked-weight recommendation

$$ \boxed{\widehat\pi_t^{\mathrm{cov},q}=\frac{m_tq_t}{1+m_tq_t}.} $$

This quantity is calculated from the known weight history rather than estimated from new random samples. In the stationary-policy submodel defined above, substituting $q_\infty(c)$ gives

$$ \boxed{\widehat\pi_\infty^{\mathrm{cov},q}=\frac c2.} $$

The equality $c/2$ describes the covariance recommendation induced by a stationary memory state. It is not the controller recursion $c_{t+1}=c_t/2$, whose only fixed point is zero. When composition varies, tracking $q_t$ costs one scalar and avoids the stationary approximation.

#### Movement premium

The covariance-only recommendation is a lower bound on the centered marginal recommendation. With $A_{0,t}:=q_tD_t^{\mathrm{old}}$ and $B_t:=D_t^{\mathrm{new}}/m_t$,

$$ \pi_{B,t}^{\mathrm{marg}}-\pi_t^{\mathrm{cov}}=\frac{S_t^{(\mathsf M)}B_t}{(A_{0,t}+B_t)(S_t^{(\mathsf M)}+A_{0,t}+B_t)}\geq0. $$

This identity makes the small-movement condition quantitative. Under equal covariance shapes, define the normalized movement premium $\rho_t:=m_tS_t^{(\mathsf M)}/D_t$. Then

$$ \boxed{\pi_{B,t}^{\mathrm{marg}}=\frac{\rho_t+m_tq_t}{\rho_t+m_tq_t+1}.} $$

The covariance contribution $m_tq_t$ is known under the equal-shape model; only $\rho_t$ requires statistical estimation. Under the ideal efficient-MLE law $K_t=\mathcal I_t^{-1}$ and population Fisher risk $\mathsf M_t=\mathcal I_t$, both covariance shapes equal $\dim\Theta$. This is an important special case, not an automatic property of finite-step deep-network optimization.

This fixed-batch rule is a local bias-variance compromise between a stable old estimate and a noisy new-batch estimate. It is not a post-optimization scaling rule: the equivalent mean-loss EWC objective uses old-to-new odds $(1-\pi_t)/\pi_t$ and accepts its optimizer solution directly.

For an oracle-recentered increment with $d\theta=b/\sqrt m$, the innovation covariance is $\pi_t^2\mathcal I^{-1}/m$. Taking $h_m=\varepsilon_m=m^{-1/2}$ gives

$$ d\Theta_t^{(\varepsilon_m)}=\pi_t b(\Theta_t^{(\varepsilon_m)})dt+\sqrt{\varepsilon_m}\,\pi_t\mathcal I(\Theta_t^{(\varepsilon_m)})^{-1/2}dW_t. $$

Its noise coefficient is linear in $\pi_t$ because the fixed set of new observations is reweighted. The finite tracking-error recursion remains the principal applied object because it retains random anchor error explicitly.

### Operational boundary

The population movement and covariance shapes in $\pi_{B,t}^{\mathrm{marg}}$ are not generally available to an online learner. The equal-shape covariance-only subproblem is exceptional because its matrix terms cancel, leaving the known scalar $q_t$. Appendix B separates this matrix-free baseline from estimation of the normalized movement premium, distinguishes a recommendation from its closed-loop application, and defines a single-trajectory calibration monitor. These constructions estimate or regularize a local surrogate; they do not promote $\pi_{B,t}^{\mathrm{marg}}$ into a globally optimal policy.


## When local optimality demands forgetting

The one-step minimizer $\pi_t^{\mathrm{loc}}$ need not be a globally desirable learning policy. A large value says that system change is large relative to local statistical uncertainty under the chosen risk. Locally, this favors rapid adaptation. Globally, the same large value also replaces more of the maintained information summary with noisy direct evidence and can erase accumulated curvature information. This coupling is part of the $\pi_t$-centric model rather than an independently tuned effect.

For the online Fisher recursion

$$ \widehat{\mathcal I}_t=(1-\pi_t)\left(\widehat{\mathcal I}_{t-1}+\widehat\Delta_t^{\mathrm{lag}}\right)+\pi_t Z_t, $$

information from an earlier estimate is weighted after $m$ constant-composition steps by approximately $(1-\pi)^m$. Under variable composition its retained weight is the realized product of the intervening $1-\pi_j$ factors. The chosen composition sequence therefore determines its own effective information horizon.

If $S_t^{(\mathsf M)}+q_tD_t^{\mathrm{old}}$ and $D_t^{\mathrm{new}}/m_t$ remain nearly constant, one fixed $\pi$ can approximate the local minimizer for a long interval. In the equal-shape fixed-batch model, the same statement is controlled by the known covariance state $m_tq_t$ and the unknown movement premium $\rho_t$. If movement, geometry, covariance shape, uncertainty, or batch size changes materially, the local minimizer can vary and an online recommendation may have value when repeated tuning trials are unavailable. This creates a reason to study adaptive composition, not a proof that any particular estimator or closed-loop policy is superior.

The MNIST experiments establish only that the tested fixed policies outperform the tested adaptive policies on their development paths. They do not establish that fixed composition is generally preferable in hardware- and data-constrained continual learning. Conversely, variation in $\pi_t^{\mathrm{loc}}$ would establish an opportunity for adaptation but not the validity of a particular recommendation mechanism.

Under this interpretation, a large local recommendation is also an overwhelm diagnostic: the one-step surrogate would require aggressive adaptation to reduce its chosen risk. It may indicate that the system is leaving the regime where a compressed local quadratic and a first-order Fisher update are reliable. Appendix B gives the predictable estimators, bounds, and calibration checks needed to operationalize this interpretation.


## Citations

[1] Y. Zheng, Y. Zhang, J. van de Weijer, G. M. van de Ven, S. Du, X. Zhang, and Z. Tian, [*Revisiting Weight Regularization for Low-Rank Continual Learning*](https://arxiv.org/abs/2602.17559), arXiv:2602.17559, 2026.

[2] S. Amari and H. Nagaoka, *Methods of Information Geometry*, American Mathematical Society, 2000.

[3] J. Kirkpatrick et al., "Overcoming catastrophic forgetting in neural networks," *Proceedings of the National Academy of Sciences*, 114(13), 3521-3526, 2017.

[4] B. A. Pearlmutter, "Fast Exact Multiplication by the Hessian," *Neural Computation*, 6(1), 147-160, 1994.

[5] S. N. Ethier and T. G. Kurtz, *Markov Processes: Characterization and Convergence*, Wiley, 1986.

[6] H. J. Kushner and G. G. Yin, *Stochastic Approximation and Recursive Algorithms and Applications*, 2nd ed., Springer, 2003.


# Appendices

## Appendix A: Diffusion and controlled small-noise limits

This appendix makes precise the scaling arguments for the theoretical Bernoulli-composition model and the applied fixed-batch model. It uses local asymptotic normality (LAN) or an asymptotically linear estimator expansion; it does not place the estimator in an exact Gaussian family, and it does not claim that LFU alone supplies the expansion. Work in one fixed parameter chart $\Theta\subseteq\mathbb R^p$, set $\sigma(\theta):=\mathcal I(\theta)^{-1/2}$, and localize to compact subsets if the coefficients are not globally bounded.

### LFU-updated information-summary assumption

At a current population solution $\theta_{k,N}^\star$, let the compressed old information be $\mathsf S_{k,N}=(\widehat\theta_{k,N},\Lambda_{k,N})$ with $\Lambda_{k,N}=N_{\mathrm{eff},k,N}\widehat{\mathcal I}_{k,N}$. The applied construction needs stronger assumptions than consistency of the Fisher estimate alone. First, after the preceding $\pi$-weighted retention and the LFU associated with a local displacement $u_{k,N}$, assume curvature calibration:

$$ \widehat{\mathcal I}_{k,N}^{\mathrm{LFU}}=\mathcal I(\theta_{k,N}^\star+u_{k,N})+o_p(1). $$

Second, let $w_{i,k,N}$ be the normalized historical influence weights, $q_{k,N}:=\sum_iw_{i,k,N}^2$, and $e_{k,N}:=\widehat\theta_{k,N}-\theta_{k,N}^\star$. Assume the centered anchor admits the local asymptotically linear representation

$$ e_{k,N}-\mu_{k,N}=\sum_iw_{i,k,N}\psi_{k,N}(X_i)+o_p(q_{k,N}^{1/2}), $$

where $\mathbb E[\psi_{k,N}\mid\mathcal G_{k,N}]=0$, $\operatorname{Cov}(\psi_{k,N}\mid\mathcal G_{k,N})=K_{k,N}+o(1)$, and a weighted Lindeberg condition rules out a dominant observation. Conditional independence, or an equivalent weak-dependence covariance condition, then gives

$$ \operatorname{Cov}(e_{k,N}\mid\mathcal G_{k,N})=q_{k,N}K_{k,N}+o_p(q_{k,N}), \qquad N_{\mathrm{eff},k,N}:=q_{k,N}^{-1}. $$

Here $\mathcal G_{k,N}$ denotes the conditioning information appropriate to the replicated local experiment. The efficient-MLE special case is $K_{k,N}=\mathcal I(\theta_{k,N}^\star)^{-1}$; the diffusion calculations below invoke that special case whenever $\mathcal I^{-1}$ appears. Neither curvature calibration nor covariance calibration follows merely from writing $\Lambda=N_{\mathrm{eff}}\widehat{\mathcal I}$, and an accurate LFU does not prove the asymptotically linear representation. Their joint use is the **LFU-updated information-summary assumption**. It must be checked after many updates or large composition weights. The language is deliberately update-based: an LFU is a coordinate-local Taylor update, not a geometric transport.

### Bernoulli composition of a local stencil

Let adjacent population solutions define the stencil displacement

$$ d\theta_{k,N}:=\theta_{k+1,N}^\star-\theta_{k,N}^\star. $$

For a local experiment of predictable size $N$, let $M_{i,k,N}$ be conditionally independent Bernoulli variables with

$$ \mathbb P(M_{i,k,N}=1\mid\mathcal F_{k,N})=\pi_{k,N}. $$

The value $M=0$ allocates a local likelihood contribution to the old point $\theta_{k,N}^\star$, represented computationally by the conditioned EWC summary, while $M=1$ allocates it to a fresh observation from the new point $\theta_{k+1,N}^\star$. Thus $\pi_{k,N}$ is the new-observation composition probability and $K_{k,N}:=\sum_{i=1}^N M_{i,k,N}$ satisfies $K_{k,N}/N\to\pi_{k,N}$ under the corresponding conditional law of large numbers. This is a local statistical construction of the mixed objective; it does not require interpreting fading as literal deletion or survival of stored observations.

### A locally consistent triangular array

For each row index $N$, let $(\mathcal F_{k,N})_{k\geq0}$ be a filtration and let $\xi_{k+1,N}$ be martingale innovations satisfying

$$ \mathbb E[\xi_{k+1,N}\mid\mathcal F_{k,N}]=0, \qquad \mathbb E[\xi_{k+1,N}\xi_{k+1,N}^T\mid\mathcal F_{k,N}]=I_p, $$

together with a conditional Lindeberg condition, uniformly over bounded time intervals. Assume $b$ and $\sigma$ are locally Lipschitz with at most linear growth, so the limiting SDEs and ODEs below are well posed. Let $\pi_{k,N}\in[0,1]$ be $\mathcal F_{k,N}$-measurable. This predictability requirement matters: a control calculated from the same innovation that generates the update generally changes the displayed conditional moments.

A useful common form for the increment is

$$ \Delta\theta_{k,N}=\pi_{k,N}b(\theta_{k,N})h_N+\sqrt{\varepsilon_N\pi_{k,N}h_N}\,\sigma(\theta_{k,N})\xi_{k+1,N}+r_{k,N}. $$

The remainders are required to be negligible at the scale of the accumulated process. For example, on every fixed horizon $T$, it is sufficient that

$$ \sup_{t\leq T}\left\|\sum_{k< t/h_N}r_{k,N}\right\|\xrightarrow{p}0, $$

with analogous $o_p(1)$ control of the accumulated conditional covariance error.

The Bernoulli stencil and LAN make the local moments explicit without assuming an exact Gaussian estimator family. Conditional on the calibrated old summary, its quadratic contributes curvature but no additional score noise. Let $e_{k,N}:=\widehat\theta_{k,N}-\theta_{k,N}^\star$ denote its realized anchor error. Regularity of the likelihood gives, uniformly for $\|e_{k,N}\|+\|d\theta_{k,N}\|=O_p(N^{-1/2})$,

$$ \mathbb E_{\theta^\star+d\theta}[s(X;\widehat\theta)]=\mathcal I(\theta^\star)(d\theta-e)+o_p(N^{-1/2}), \qquad \operatorname{Var}_{\theta^\star+d\theta}[s(X;\widehat\theta)]=\mathcal I(\theta^\star)+o_p(1). $$

Consequently, the $K_{k,N}$ new scores have the LAN expansion

$$ S_{k,N}=K_{k,N}\mathcal I(\theta_{k,N}^\star)(d\theta_{k,N}-e_{k,N})+\sqrt{K_{k,N}}\,\mathcal I(\theta_{k,N}^\star)^{1/2}\xi_{k+1,N}+o_p(\sqrt N). $$

The old quadratic and new likelihood together have local negative Hessian $N\mathcal I(\theta_{k,N}^\star)+o_p(N)$. One Newton step, or an asymptotically equivalent local MLE, therefore satisfies

$$ \widehat\theta_{k+1,N}-\theta_{k,N}^\star=e_{k,N}+\frac{K_{k,N}}{N}(d\theta_{k,N}-e_{k,N})+\frac{\sqrt{K_{k,N}}}{N}\mathcal I(\theta_{k,N}^\star)^{-1/2}\xi_{k+1,N}+o_p(N^{-1/2}). $$

Since $K_{k,N}/N\to\pi_{k,N}$, the conditional mean relative to $\theta_{k,N}^\star$ is $e_{k,N}+\pi_{k,N}(d\theta_{k,N}-e_{k,N})+o_p(N^{-1/2})$ and the conditional covariance is $\pi_{k,N}\mathcal I^{-1}/N+o_p(N^{-1})$. Hence the conditional squared bias relative to the next true solution is $(1-\pi_{k,N})^2\|d\theta_{k,N}-e_{k,N}\|^2$. The simpler risk below sets $e_{k,N}=0$, equivalently using an oracle-recentered current solution. One may instead retain the same algebra by defining its signal as the vector from the realized anchor to the next true solution, $d\theta_{k,N}-e_{k,N}$. Unconditional risk across trajectories must average over anchor uncertainty and any cross-covariance with the new score.

### Ordinary fixed-composition diffusion

Take a fixed $\pi\in[0,1]$, $h_N=N^{-1}$, and $\varepsilon_N=1$. Then

$$ \mathbb E[\Delta\theta_{k,N}\mid\mathcal F_{k,N}]=\pi b(\theta_{k,N})h_N+o_p(h_N), $$

$$ \operatorname{Cov}(\Delta\theta_{k,N}\mid\mathcal F_{k,N})=\pi\mathcal I(\theta_{k,N})^{-1}h_N+o_p(h_N). $$

Let $\Theta^N$ be the piecewise-constant or polygonal interpolation with $k=\lfloor t/h_N\rfloor=\lfloor Nt\rfloor$. The drift characteristics converge to $\int_0^t\pi b(\Theta_s)ds$, the predictable quadratic variation converges to $\int_0^t\pi\mathcal I(\Theta_s)^{-1}ds$, and the conditional Lindeberg condition excludes macroscopic jumps. Standard martingale-problem or diffusion-approximation results [5,6] therefore give

$$ \Theta^N\Rightarrow\Theta \quad\text{in }D([0,T],\mathbb R^p), $$

where the unique weak solution satisfies

$$ \boxed{d\Theta_t=\pi b(\Theta_t)dt+\sqrt\pi\,\mathcal I(\Theta_t)^{-1/2}dW_t.} $$

The factor $\sqrt\pi$ is forced by quadratic variation. For constant $\pi$, the generator is $\mathcal L_\pi=\pi\mathcal L$, so the process is the base learning diffusion run on the slower clock $\pi t$.

### Why stochastic control uses the LAN scale

The one-step decision problem minimizes mean-squared error to the next population solution, not an error to a thinned dataset. At stencil index $k$, its estimand is

$$ \mathcal R_{k,N}(\pi):=\mathbb E\left[\left\|\widehat\theta_{k+1,N}-(\theta_{k,N}^\star+d\theta_{k,N})\right\|^2\mid\mathcal F_{k,N}\right]. $$

Thus $d\theta_{k,N}$ is concretely the vector from the current true solution point to the next true solution point at the chosen stencil granularity. It is an oracle quantity in the experiment unless a separate estimator is supplied. The displayed risk now adopts the oracle-recentered case $e_{k,N}=0$; otherwise replace $d\theta_{k,N}$ by $d\theta_{k,N}-e_{k,N}$ in its conditional bias term. If the environmental displacement were $b(\theta)/N$, its squared magnitude would be $O(N^{-2})$ while local estimator variance would be $O(N^{-1})$; the asymptotic one-step rule would collapse to no adaptation. If the displacement remained $O(1)$, variance would vanish and the rule would collapse to full adaptation. The nondegenerate LAN modeling assumption is

$$ d\theta_{k,N}=\frac{b(\theta_{k,N}^\star)}{\sqrt N}. $$

Here $b$ is the limiting vector field, in the sense that $\sqrt N\,d\theta_{k,N}\to b(\theta)$ along the triangular array. This scaling is a chosen local asymptotic regime, not a causal relationship between information and environmental movement. Under the conditional LAN expansion above, an update with composition $\pi_{k,N}$ has conditional mean $\pi_{k,N}b/\sqrt N$ and covariance $\pi_{k,N}\mathcal I^{-1}/N$. Its one-step Euclidean risk is

$$ \frac1N\left[(1-\pi)^2\|b(\theta)\|^2+\pi\operatorname{tr}\mathcal I(\theta)^{-1}\right]+o(N^{-1}), $$

so the limiting optimization over $\pi$ is nondegenerate. For known $b$ and interior solutions it gives

$$ \pi_A^{\mathrm{loc}}(\theta)=1-\frac{\operatorname{tr}\mathcal I(\theta)^{-1}}{2\|b(\theta)\|^2}, $$

followed by clipping to $[0,1]$. Equivalently, before substituting $d\theta=b/\sqrt N$, this is the finite-$N$ expression involving $N\|d\theta\|^2$. The formula is conditional on Euclidean loss, the LAN local experiment, calibration of the old summary, and knowledge of the environmental displacement; changing any of these changes the local minimizer.

A time-varying effective size can replace $N$ locally. In particular, $N_{t-1}$ may be used in the one-step formula when it is predictable, diverges along the asymptotic sequence, calibrates the summary covariance as assumed above, and is held fixed while optimizing the next composition $\pi_t$. The associated local alternative is then $d\theta_t=b(\theta_t)/\sqrt{N_{t-1}}+o(N_{t-1}^{-1/2})$. This is a rigorous predictable plug-in construction. It is not the same experiment if choosing $\pi_t$ also changes the denominator through an identity such as $N_t=m_t/\pi_t$ for a fixed new batch size $m_t$; in that case the variance is proportional to $\pi_t^2/m_t$ and the risk must be re-optimized jointly rather than importing the fixed-$N$ formula.

### Controlled fluid limit

Set $h_N=N^{-1/2}$ and suppose the predictable controls $\pi_{k,N}$ converge in a mode sufficient for their drift Riemann sums to converge to a predictable process $\pi_t$. The controlled LAN recursion has the form

$$ \Delta\theta_{k,N}=\pi_{k,N}b(\theta_{k,N})h_N+\sqrt{\pi_{k,N}}\,\sigma(\theta_{k,N})h_N\xi_{k+1,N}+r_{k,N}. $$

Over $\lfloor T/h_N\rfloor=O(\sqrt N)$ steps, the drift is $O(1)$ but the accumulated conditional covariance is only

$$ \sum_{k<T/h_N}\pi_{k,N}\mathcal I(\theta_{k,N})^{-1}h_N^2=O(h_N)=O(N^{-1/2}). $$

A martingale maximal inequality therefore makes the stochastic term vanish uniformly in probability, while the drift converges to its Riemann integral. Subject to the stated remainder and control-convergence assumptions,

$$ \Theta^N\xrightarrow{p}\Theta^0, \qquad \Theta_t^0=\Theta_0^0+\int_0^t\pi_s b(\Theta_s^0)ds, $$

or

$$ \boxed{d\Theta_t^0=\pi_t b(\Theta_t^0)dt.} $$

### Finite-information small-noise diffusion

The deterministic fluid limit suppresses finite-$N$ estimator variability. To retain its leading local covariance, set

$$ \varepsilon_N:=h_N=N^{-1/2} $$

and consider the controlled diffusion

$$ \boxed{d\Theta_t^{(\varepsilon_N)}=\pi_t b(\Theta_t^{(\varepsilon_N)})dt+\sqrt{\varepsilon_N\pi_t}\,\mathcal I(\Theta_t^{(\varepsilon_N)})^{-1/2}dW_t.} $$

Over one Euler interval of length $h_N$, its conditional drift is $\pi_tb h_N$ and its conditional covariance is

$$ \varepsilon_N\pi_t\mathcal I^{-1}h_N=\pi_t\mathcal I^{-1}h_N^2=\frac{\pi_t}{N}\mathcal I^{-1}, $$

which matches the controlled LAN recursion. Equivalently, the diffusion noise coefficient is $O(N^{-1/4})$ and the Brownian increment over one $N^{-1/2}$ interval is also $O(N^{-1/4})$, producing the required $O(N^{-1/2})$ discrete noise. Thus the small-noise SDE is the finite-information diffusion interpolation of the controlled chain, while $\Theta^0$ is its $N\to\infty$ fluid limit.

The ordinary and controlled displays are therefore two regimes of the same local-moment template:

| regime | $h_N$ | $\varepsilon_N$ | accumulated quadratic variation |
|---|---:|---:|---:|
| fixed-composition diffusion | $N^{-1}$ | $1$ | $O(1)$ |
| controlled LAN / small noise | $N^{-1/2}$ | $N^{-1/2}$ | $O(N^{-1/2})$ |

### Fixed-batch stratified local experiment

The Bernoulli construction fixes the total stencil size and lets the number of new observations vary. The applied experiment instead receives a fixed new batch of size $m$ and uses the old summary as a separately represented stratum. Under LAN at the new population point,

$$ \widehat\theta_{\mathrm{new},k,m}=\theta_{k+1,m}^\star+\frac1{\sqrt m}\mathcal I(\theta_{k,m}^\star)^{-1/2}\xi_{k+1,m}+o_p(m^{-1/2}). $$

Locally matched quadratic curvature and the mixture-weighted EWC objective give

$$ \widehat\theta_{k+1,m}=(1-\pi_{k,m})\widehat\theta_{k,m}+\pi_{k,m}\widehat\theta_{\mathrm{new},k,m}+o_p(m^{-1/2}). $$

Writing $e_{k,m}:=\widehat\theta_{k,m}-\theta_{k,m}^\star$ and $d\theta_{k,m}:=\theta_{k+1,m}^\star-\theta_{k,m}^\star$ yields

$$ e_{k+1,m}=(1-\pi_{k,m})(e_{k,m}-d\theta_{k,m})+\frac{\pi_{k,m}}{\sqrt m}\mathcal I(\theta_{k,m}^\star)^{-1/2}\xi_{k+1,m}+o_p(m^{-1/2}). $$

Suppose the old summary is covariance calibrated with effective size $N_{k,m}$, the old error and new innovation are locally centered and conditionally independent, and $d\theta_{k,m}=b(\theta_{k,m}^\star)/\sqrt m$. Then, with $T(\theta):=\operatorname{tr}\mathcal I(\theta)^{-1}$,

$$ m\,\mathbb E\|e_{k+1,m}\|^2=(1-\pi)^2\left(\|b(\theta)\|^2+\frac{m}{N_{k,m}}T(\theta)\right)+\pi^2T(\theta)+o(1). $$

If $m/N_{k,m}$ converges locally, the limiting risk is nondegenerate and has minimizer

$$ \boxed{\pi_B^{\mathrm{loc}}(\theta)=\frac{\|b(\theta)\|^2+[m/N_{k,m}]T(\theta)}{\|b(\theta)\|^2+[m/N_{k,m}]T(\theta)+T(\theta)}.} $$

Undoing the LAN substitution gives the finite-step formula in the main text with $\tau_{\mathrm{old}}=T/N_{\mathrm{eff}}$ and $\tau_{\mathrm{new}}=T/m$. This local minimizer is marginal over calibrated old-anchor error. Conditional on a realized old anchor, replace $d\theta$ by the vector from that anchor to the next true solution and omit $\tau_{\mathrm{old}}$ from the numerator. In the zero-movement case, substituting $q=N_{\mathrm{eff}}^{-1}$ reduces the efficient fixed-batch minimizer to $mq/(1+mq)$, confirming the scalar covariance-only result within the LAN construction.

For the oracle-recentered innovation, set $h_m=\varepsilon_m=m^{-1/2}$ and write

$$ \Delta\theta_{k,m}=\pi_{k,m}b(\theta_{k,m})h_m+\pi_{k,m}\mathcal I(\theta_{k,m})^{-1/2}h_m\xi_{k+1,m}+r_{k,m}. $$

The matching finite-information interpolation is

$$ \boxed{d\Theta_t^{(\varepsilon_m)}=\pi_t b(\Theta_t^{(\varepsilon_m)})dt+\sqrt{\varepsilon_m}\,\pi_t\mathcal I(\Theta_t^{(\varepsilon_m)})^{-1/2}dW_t.} $$

One Euler interval has covariance $\varepsilon_mh_m\pi_t^2\mathcal I^{-1}=\pi_t^2\mathcal I^{-1}/m$. Its accumulated quadratic variation over $O(\sqrt m)$ controlled LAN steps is $O(m^{-1/2})$, so it has the same deterministic fluid limit as the Bernoulli model but a different finite-information noise law. The applied EWC recursion for $e_{k,m}$ remains more informative than this interpolation because it retains the serial dependence induced by the random anchor.

The two models meet pointwise when a Bernoulli experiment happens to use $m=\pi N$: then $\pi^2/m=\pi/N$. They are nevertheless different control experiments because the applied model holds $m$ fixed while optimizing $\pi$, whereas the theoretical model holds $N$ fixed and lets the new count vary.

| model | fixed quantity | new-score covariance after weighting | small-noise coefficient | role |
|---|---:|---:|---:|---|
| Bernoulli composition | total $N$ | $\pi\mathcal I^{-1}/N$ | $\sqrt{\varepsilon\pi}\,\mathcal I^{-1/2}$ | theory oracle |
| fixed-batch EWC | new batch $m$ | $\pi^2\mathcal I^{-1}/m$ | $\sqrt\varepsilon\,\pi\mathcal I^{-1/2}$ | principal applied model |

This construction still leaves empirical assumptions to validate. In particular, the experiment must test the LFU-updated information-summary calibration, local constancy of the population trend over the chosen half-life, weak enough serial dependence for the residual moment estimator, and fidelity of the optimizer to the local quadratic mixture. Predictable one-step-lagged recommendation removes same-observation selection bias. These are operational assumptions, not consequences of the LFU identity.


## Appendix B: Statistical and numerical realization

The main text defines population one-step risks and their local surrogate minimizers. This appendix describes what an online learner can estimate, how exponentially discounted risk (EDR) regularizes those estimates, and what can be diagnosed from one realized trajectory. These constructions are operational accommodations, not additional optimality results.

### Population Fisher and its predictable representation

The population Fisher $\mathcal I_t:=\mathcal I(\theta_t^\star)$ is the metric and covariance operator in the ideal model. The inverse-based displays in the main text and Appendix A assume it is nonsingular. The online learner instead stores a predictable estimate

$$ G_t:=\widehat{\mathcal I}_{t\mid t-1}\succeq0, $$

constructed before the step-$t$ observation batch. A finite-sample $G_t$ may be singular even when the theoretical Fisher is nonsingular. Fisher-risk calculations require only quadratic products $v^TG_tv$; they do not invert or pseudoinvert $G_t$. Low-rank-plus-diagonal and diagonal representations are therefore admissible provided they implement the same predictable quadratic.

The EWC summary treats $N_{\mathrm{eff},t}G_t$ as retained local precision. This is stronger than Fisher consistency alone. Appendix A formalizes covariance calibration as $\operatorname{Cov}(e_t\mid\mathcal E_t)=q_tK_t+o(q_t)$, where $q_t=N_{\mathrm{eff},t}^{-1}$ is exact weight concentration but $K_t$ is an assumed local covariance shape. The efficient relation $K_t=\mathcal I_t^{-1}$ is a special case. Experiments must check the covariance approximation rather than infer it from the notation.

### Matrix-free covariance-only recommendation

Using the main text's fixed-batch timing, $q_t$ is known before the learner chooses $\pi_t$. The statistical realization separates three levels. The population covariance oracle is

$$ \pi_t^{\mathrm{cov}}=\frac{q_tD_t^{\mathrm{old}}}{q_tD_t^{\mathrm{old}}+D_t^{\mathrm{new}}/m_t}. $$

If the old and new local covariance shapes match in the chosen risk metric, their common unknown scale cancels and yields the matrix-free recommendation

$$ \boxed{\widehat\pi_t^{\mathrm{cov},q}=\frac{m_tq_t}{1+m_tq_t}.} $$

The learner calculates this value from the recursively maintained $q_t$; it does not estimate $q_t$ from score samples. In the stationary-policy submodel, substituting the main text's $q_\infty(c)$ gives the approximation $c/2$. Its error is structural rather than ordinary sampling error. It is not a policy update, and under varying actions the tracked-$q_t$ form remains the appropriate calculation.

This simplification is specific to the applied fixed-batch variance law $\pi_t^2K_t/m_t$. The theoretical Bernoulli-composition model changes the number of random scores and has variance linear in $\pi_t$, so it does not imply this recommendation.

### Predictable movement plug-in recommendation

The deployable construction uses the accepted EWC trajectory rather than a disposable small-batch MLE. Under matched local quadratic curvature,

$$ u_t:=\widehat\theta_{t+1}-\widehat\theta_t\approx\pi_t(d\theta_t+\xi_t-e_t). $$

For $\pi_t>0$, define the normalized observation $z_t=u_t/\pi_t$. With a predictable gain $\gamma_t$ on the chosen update clock, maintain

$$ \widehat d_{t+1\mid t}=(1-\gamma_t)\widehat d_{t\mid t-1}+\gamma_tz_t. $$

Freeze $G_t$, the pre-transition concentration $q_t$, and $\widehat d_{t\mid t-1}$ before observing the accepted step. Define

$$ r_t:=u_t-\pi_t\widehat d_{t\mid t-1}, \qquad a_t:=\pi_t^2\left(q_t+m_t^{-1}\right), \qquad E_t:=r_t^TG_tr_t. $$

Under the equal-shape calibrated model, an accurate population-trend predictor, and the replicated-local-experiment conditioning $\mathcal E_t$, $\mathbb E(E_t\mid\mathcal E_t)\approx a_tD_t$, where $D_t=\operatorname{tr}(G_tK_t)$ for the common covariance shape $K_t$. This is not generally an expectation conditional on the operational history $\mathcal F_{t-1}$: there the current anchor error is fixed rather than a centered random draw. The online construction substitutes local temporal averaging for unavailable replicated learners, requiring enough local stability or ergodicity that old residuals mimic the stipulated replicated moment. Subject to that additional approximation, estimate the scalar without a Fisher inverse by maintaining matched predictable moments

$$ M_t^E=(1-\gamma_t)M_{t-1}^E+\gamma_tE_t, \qquad M_t^a=(1-\gamma_t)M_{t-1}^a+\gamma_ta_t, \qquad \widehat D_{t+1\mid t}=\frac{M_t^E}{M_t^a+\varepsilon}. $$

In the efficient special case $K_t=\mathcal I_t^{-1}$, choosing $G_t=I_{\dim\Theta}$ gives $D_t=\operatorname{tr}\mathcal I_t^{-1}$ and recovers the Euclidean trace estimator, while $G_t=\mathcal I_t$ gives $D_t=\dim\Theta$. With a general covariance shape and estimated Fisher, $\widehat D$ instead targets the common Fisher-weighted covariance risk actually seen by the maintained quadratic.

At decision time, the instantaneous predictable coefficients are

$$ \widehat A_{t\mid t-1}=\widehat d_{t\mid t-1}^TG_t\widehat d_{t\mid t-1}+q_t\widehat D_{t\mid t-1}, \qquad \widehat B_{t\mid t-1}=\frac{\widehat D_{t\mid t-1}}{m_t}. $$

Their instantaneous plug-in recommendation is

$$ \widetilde\pi_t^{\mathrm{rec}}=\frac{\widehat A_{t\mid t-1}}{\widehat A_{t\mid t-1}+\widehat B_{t\mid t-1}}. $$

Under the equal-shape model, define the estimated normalized movement premium

$$ \widehat\rho_{t\mid t-1}:=\frac{m_t\widehat d_{t\mid t-1}^TG_t\widehat d_{t\mid t-1}}{\widehat D_{t\mid t-1}}. $$

Then the same instantaneous recommendation can be written

$$ \boxed{\widetilde\pi_t^{\mathrm{rec}}=\frac{\widehat\rho_{t\mid t-1}+m_tq_t}{\widehat\rho_{t\mid t-1}+m_tq_t+1}.} $$

This form isolates the exact scalar covariance baseline $m_tq_t$ from the statistically difficult movement premium. It is causal only because every term is available before the batch whose objective it weights. The interpretation remains delicate: the accepted-trajectory trend may estimate population movement $d\theta_t$, realized catch-up $d\theta_t-e_t$, or a mixture under serial optimizer dynamics. Historical software and immutable artifacts call this treatment `optimal_plugin`; that name means plug-in minimization of the assumed local surrogate, not demonstrated global or empirical optimality.

### Exponentially discounted recommendation

The instantaneous coefficients are noisy. EDR applies an accepted-update gain $\eta=1-2^{-1/H_\pi}$ to the coefficients themselves:

$$ \overline A_t=(1-\eta)\overline A_{t-1}+\eta\widehat A_{t\mid t-1}, \qquad \overline B_t=(1-\eta)\overline B_{t-1}+\eta\widehat B_{t\mid t-1}, $$

$$ \boxed{\pi_t^{\mathrm{EDR,rec}}=\frac{\overline A_t}{\overline A_t+\overline B_t}.} $$

When the population coefficients are locally constant, discounting is primarily variance reduction. Under nonstationarity, $(\overline A_t,\overline B_t)$ target a temporally weighted surrogate rather than the instantaneous $(A_t,B_t)$. EDR consequently trades noise reduction for lag bias; recommendation hysteresis is one observable consequence. A fixed cold-start action may be used while the trend and residual moments acquire support.

The scalar decomposition reveals that historical EDR smooths more than is necessary in the equal-shape covariance-only regime: $q_t$ is already known exactly. A future decomposed treatment could retain the current $m_tq_t$ and discount only a nonnegative movement estimate, for example

$$ \overline\rho_t=(1-\eta)\overline\rho_{t-1}+\eta\widehat\rho_{t\mid t-1}, \qquad \pi_t^{\mathrm{decomp,rec}}=\frac{\overline\rho_t+m_tq_t}{\overline\rho_t+m_tq_t+1}. $$

This is a proposed numerical treatment, not an equivalent rewriting of historical EDR when $q_t$, $m_t$, or covariance shape varies. It has not been applied to the completed artifacts. The existing EDR equations and artifact names are retained so past evidence is not silently reinterpreted.

### Recommendation and closed-loop application

A recommendation is a predictable statistic. Applying it creates a policy. Let $\pi_t^{\mathrm{used}}$ denote the value actually supplied to both the EWC objective and the Fisher recursion. It changes the accepted displacement, $q_{t+1}$, the future Fisher summary, the future trend and residual moments, and therefore every later recommendation. Thus a same-state reduction in the estimated quadratic does not imply lower cumulative predictive loss on the resulting closed-loop path.

Ordinary actions use

$$ \pi_t^{\mathrm{used}}=\min\!\left(\pi_{\max},\max\!\left(\pi_{\min},\pi_t^{\mathrm{rec}}\right)\right), \qquad 0<\pi_{\min}\leq\pi_{\max}\leq1. $$

The lower bound preserves EWC odds, residual excitation, and recovery after quiet intervals; the upper bound limits abrupt adaptation and near-total replacement of the Fisher summary. A deliberate hard freeze is a separate action. Boundary choices are safeguards and introduce their own bias; they are not part of the population minimizer.

### Single-trajectory prequential calibration

The lagged scalar $\widehat D_{t\mid t-1}$ predicts the residual Fisher-risk energy

$$ P_t:=a_t\widehat D_{t\mid t-1}, $$

before $E_t=r_t^TG_tr_t$ is observed. A causal calibration monitor on the EDR action timescale is

$$ \boxed{C_t:=\frac{\operatorname{EMA}_{H_C}(E_t)}{\operatorname{EMA}_{H_C}(P_t)}.} $$

$C_t=1$ is locally calibrated, $C_t>1$ means residual risk was underpredicted, and $C_t<1$ means it was overpredicted. Plotting $\log C_t$ makes the two directions symmetric. Plan 4 uses $H_C=4$ accepted updates, matching its EDR action timescale. The monitor uses one trajectory and no fixed-policy comparator, but it is not an independent goodness-of-fit test: $\widehat D$ is itself learned from older residuals. Its role is to reveal short-horizon staleness relative to the slower uncertainty estimate. It cannot certify that applying the recommendation improves predictive performance.

### Experimental status

The MNIST experiments support a rank-8-plus-diagonal direct-EMA Fisher summary without LFUs as the practical baseline. The tested LFU corrections were too noisy and projection-heavy to justify their cost; this is an applied result, not a contradiction of the LFU identity. Fixed $\pi=.05$ outperformed the tested adaptive policies on these paths, but that is a conditional MNIST result rather than a general theorem favoring fixed composition.

Plan 7 isolates a stronger result for the applied fixed-batch covariance-only subproblem. On the stored rank-16 EDR trajectory, $m_tq_t/(1+m_tq_t)$ reproduces the high-sample covariance oracle with mean absolute error `.0008` on the linear schedule and `.0009` on the sigmoid schedule. The pointwise local-stationary approximation $\pi_t^{\mathrm{used}}/2$ has corresponding errors `.0094` and `.0120`; its theoretical constant-action version is $c/2$. Adding estimated population movement raises the recommendation by only about `.0026` on either path. These matched, single-trajectory findings support the scalar recommendation in this regime; they do not establish universal covariance calibration or predictive superiority.

EDR extracted structured online signal: on the linear path it moved from a $.05$ cold start toward the hindsight-useful $.025$ region, and the prequential $C_t$ monitor exposed increasingly stale risk estimates after sharper sigmoid transitions. Its realized closed-loop paths nevertheless underperformed the fixed comparators. The new decomposition shows that the difficult statistical problem lies chiefly in movement and realized-anchor effects, not in reconstructing the equal-shape covariance baseline. EDR therefore remains an unconfirmed estimator of that additional signal. Whether a decomposed adaptive rule helps when the locally appropriate $\pi_t$ changes materially and repeated tuning trials are unavailable remains open.
